# NagarMitra: a local-first civic action agent powered by Gemma 4

This notebook is the reproducible model-backed path for the Build with Gemma: TFUG Prayagraj hackathon. Gemma 4 reads a resident report (and optionally an image), calls allow-listed local civic tools, and returns a human-reviewable bilingual action card. It never submits a complaint.

In [ ]:
# Kaggle provides a matched torch/torchvision/Pillow stack. Keep those versions intact.
!pip -q install -U accelerate 'transformers>=4.51'

from pathlib import Path
import os, sys

repo = Path('/kaggle/working/nagarmitra-gemma4')
if not repo.exists():
    !git clone https://github.com/yinli-systems/nagarmitra-gemma4.git /kaggle/working/nagarmitra-gemma4
sys.path.insert(0, str(repo))
print('Source:', repo)

In [ ]:
from gemma4_agent import TOOL_DECLARATIONS, extract_tool_calls, execute_allowlisted

# A local smoke test runs before any model load. Unknown tools are rejected.
sample = '<|tool_call>call:lookup_department{issue_type:<|"|>garbage_overflow<|"|>}<tool_call|>'
calls = extract_tool_calls(sample)
assert calls[0]['name'] == 'lookup_department'
assert execute_allowlisted(calls)[0]['response']['ok'] is True
print('Allow-list smoke test passed:', calls)
print('Tools exposed to Gemma 4:', [x['function']['name'] for x in TOOL_DECLARATIONS])

In [ ]:
# Resolve the attached Kaggle model through KaggleHub, as shown in the
# official Gemma 4 model example; this also handles the mounted version path.
MODEL_REF = 'google/gemma-4/transformers/gemma-4-e2b-it'
MODEL_ID = os.environ.get('GEMMA_MODEL_ID')
if not MODEL_ID:
    import kagglehub
    MODEL_ID = kagglehub.model_download(MODEL_REF)
print('Resolved Gemma 4 model:', MODEL_ID)
from gemma4_agent import Gemma4Agent

agent = Gemma4Agent(model_id=MODEL_ID)
result = agent.run(
    report='कूड़ा तीन दिन से नहीं उठा है और नाली बंद हो रही है।',
    place='Naini, near school gate',
    language='hi',
)
print('Tool trace:')
for row in result['trace']:
    print('-', row['tool'], row['response'].get('ok'))
print('\nGemma 4 answer:\n', result['answer'])

## Evidence boundary

The static GitHub Pages presentation uses deterministic fixture records so judges can inspect the full UX without waiting for a model download. The execution above is the actual Gemma 4 path; the two are intentionally labelled separately.